# Linear Regression From Scratch

Notebook for **Chapter 2** of the *ml* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/ml/ch02.html

Tip: change a number, predict the output, then run the cell again.

## 2.1 The Model and the Normal Equation

In [ ]:
import numpy as np
x = np.array([1, 2, 3.]); y = np.array([2, 3, 5.])
X = np.c_[np.ones(3), x]
print("XtX\n", X.T @ X, "\nXty", X.T @ y)
beta = np.linalg.solve(X.T @ X, X.T @ y); print("beta", beta)
res = y - X @ beta; print("resid", res, "X^T resid", X.T @ res)

## 2.2 Gradient Descent Version

In [ ]:
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

class MyLinearRegression:
    """OLS two ways: normal equation or gradient descent."""
    def __init__(self, method="normal", lr=0.1, steps=20000):
        self.method, self.lr, self.steps = method, lr, steps
    def fit(self, X, y):
        A = np.c_[np.ones(len(X)), X]                      # add the intercept column of ones
        if self.method == "normal":
            self.w = np.linalg.solve(A.T @ A, A.T @ y)     # solve (XtX) w = Xty
        else:
            self.w = np.zeros(A.shape[1]); n = len(y)
            for _ in range(self.steps):
                grad = 2 / n * A.T @ (A @ self.w - y)      # gradient of the mean squared error
                self.w -= self.lr * grad
        return self
    def predict(self, X):
        return np.c_[np.ones(len(X)), X] @ self.w

d = load_diabetes()
Xtr, Xte, ytr, yte = train_test_split(d.data, d.target, test_size=0.25, random_state=0)
mu, sd = Xtr.mean(0), Xtr.std(0)
Ztr, Zte = (Xtr - mu) / sd, (Xte - mu) / sd               # scale with TRAIN statistics only
rmse = lambda a, p: np.sqrt(np.mean((a - p) ** 2))
ne = MyLinearRegression("normal").fit(Ztr, ytr)
gd = MyLinearRegression("gd").fit(Ztr, ytr)
sk = LinearRegression().fit(Ztr, ytr)
print("intercepts :", round(ne.w[0], 4), round(gd.w[0], 4), round(sk.intercept_, 4))
print("max |coef diff| NE vs sklearn:", np.abs(ne.w[1:] - sk.coef_).max())
print("max |coef diff| GD vs NE     :", np.abs(gd.w - ne.w).max())
print("test RMSE  NE/GD/sklearn:", [round(rmse(yte, m.predict(Zte)), 3) for m in (ne, gd, sk)])
print("baseline (predict train mean) RMSE:", round(rmse(yte, np.full(len(yte), ytr.mean())), 3))
print("test R2:", round(1 - np.sum((yte - ne.predict(Zte))**2) / np.sum((yte - yte.mean())**2), 4))

In [ ]:
rng = np.random.default_rng(0)
x1 = rng.normal(0, 1, 200); x2 = rng.normal(0, 1000, 200)
yy = 3 * x1 + 0.002 * x2 + rng.normal(0, .5, 200)
Xs = np.c_[np.ones(200), x1, x2]
lam = np.linalg.eigvalsh(2/200 * Xs.T @ Xs); print("eig", lam, "max safe lr", 2 / lam[-1])

eig [      1.8375       1.9878 2118086.5722] max safe lr 9.442484675705359e-07


## 2.3 Assumptions and What Breaks Them

In [ ]:
x = rng.uniform(1, 10, 300); y = 2*x + rng.normal(0, 0.3*x**1.5)
A = np.c_[np.ones(300), x]; b = np.linalg.lstsq(A, y, rcond=None)[0]; r = y - A@b
print("coef", b)
for lo, hi in [(1, 4), (4, 7), (7, 10)]:
    m = (x >= lo) & (x < hi); print("x in", lo, hi, "resid std", r[m].std().round(2))

coef [-0.7546  2.1163]
x in 1 4 resid std 1.26
x in 4 7 resid std 3.6
x in 7 10 resid std 8.2


## 2.4 Inference vs Prediction: Reading Coefficients Honestly

## 2.5 Curves, Interactions and Interpretable Non-Linearity

In [ ]:
from sklearn.preprocessing import PolynomialFeatures, SplineTransformer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
xs = np.sort(rng.uniform(0, 1, 60)); ys = np.sin(2*np.pi*xs) + rng.normal(0, .3, 60)
xt = np.linspace(0, 1, 500)[:, None]; yt_ = np.sin(2*np.pi*xt[:, 0])
for name, m in [("linear", LinearRegression()),
                ("poly3", make_pipeline(PolynomialFeatures(3), LinearRegression())),
                ("poly15", make_pipeline(PolynomialFeatures(15), LinearRegression())),
                ("spline(6 knots,deg3)", make_pipeline(SplineTransformer(n_knots=6, degree=3), LinearRegression()))]:
    m.fit(xs[:, None], ys); print(name, "test MSE vs truth", np.mean((m.predict(xt) - yt_)**2).round(4))

linear test MSE vs truth 0.1998
poly3 test MSE vs truth 0.0111
poly15 test MSE vs truth 0.0487
spline(6 knots,deg3) test MSE vs truth 0.0097


## 2.6 Bayesian Linear Regression and Gaussian Processes: Predictions With Honest Error Bars

Ordinary least squares gives one line. If you only have 8 points, you know the line is uncertain, and you know it is *more* uncertain far from the data. **Bayesian linear regression** keeps the whole family of plausible lines: put a prior on the weights (a bell curve centred at zero with precision `α`), assume Gaussian noise with variance `σ²`, and the posterior is again a bell curve with

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
x = np.linspace(-3, 3, 8); y = 1.0 + 2.0 * x + rng.normal(0, 1.0, 8)
Phi = np.c_[np.ones_like(x), x]; sig2, alpha = 1.0, 1.0       # noise var, prior precision
S = np.linalg.inv(alpha * np.eye(2) + Phi.T @ Phi / sig2); m = S @ Phi.T @ y / sig2
ridge = np.linalg.solve(Phi.T @ Phi + alpha * sig2 * np.eye(2), Phi.T @ y)
print("posterior mean", m.round(3), "ridge (lambda=alpha*sigma2)", ridge.round(3))
print("posterior sd", np.sqrt(np.diag(S)).round(3))
for xs in (0.0, 2.0, 6.0):
    f = np.array([1, xs]); print(f"x={xs}: pred {f@m:.2f}  sd of mean {np.sqrt(f@S@f):.2f}  predictive sd {np.sqrt(f@S@f+sig2):.2f}")
from sklearn.gaussian_process import GaussianProcessRegressor as GPR
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel as C
xt = np.sort(rng.uniform(-4, 4, 12)); yt = np.sin(xt) + rng.normal(0, 0.15, 12)
gp = GPR(C(1.0) * RBF(1.0) + WhiteKernel(0.1), normalize_y=True, n_restarts_optimizer=5, random_state=0).fit(xt[:, None], yt)
print(gp.kernel_)
mu, sd = gp.predict(np.array([[0.0], [2.0], [8.0]]), return_std=True)
print("mean", mu.round(2), "sd", sd.round(2))

posterior mean [1.202 2.091] ridge (lambda=alpha*sigma2) [1.202 2.091]
posterior sd [0.333 0.177]
x=0.0: pred 1.20  sd of mean 0.33  predictive sd 1.05
x=2.0: pred 5.38  sd of mean 0.49  predictive sd 1.11
x=6.0: pred 13.75  sd of mean 1.11  predictive sd 1.50
1.41**2 * RBF(length_scale=1.4) + WhiteKernel(noise_level=0.0322)
mean [0.04 0.78 0.11] sd [0.12 0.13 0.8 ]


## Chapter recap

## Practice set

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/ml/ch02.html